# W6D3 — Agentic AI with LangGraph: build a real small agent — Lab

**Week 6 · Day 3 · LLMs** · Lab


> **The big idea.** A plain LLM has a **mouth** — it can talk about anything,
> and get the arithmetic wrong. An **agent** has a mouth **and hands**.
> The hands are tools: real Python functions (`calculator`, `kb_lookup`, `search_web`).
> **LangGraph** is how production teams connect the mouth to the hands — a small
> graph of nodes you wire together once, then invoke on any question:
>
> ```
>              Question
>                 |
>                 v
>        +----------------+
>    .-->|     THINK      |  (LLM decides: use a tool, or answer?)
>    |   +-------+--------+
>    |           |
>    |           |  "Action: calculator"
>    |           |  "Action Input: 40 * 2500 / 3.75"
>    |           v
>    |   +----------------+
>    |   |      ACT       |  (Python runs the tool, returns a string)
>    |   +-------+--------+
>    |           |
>    |           |  "Observation: 26666.67"
>    `-----------+
>                |
>                |  (loop until the LLM says "Final Answer: ...")
>                v
>              Answer
> ```
>
> Today you build this exact graph in **LangGraph** (Section 2), then level up
> to a **multi-agent system** where a *supervisor* routes each question to a
> *specialist* that owns one tool (Section 3). That is the pattern every
> production team ends up using.

<div dir="rtl" align="right">

> **الفكرة الكبيرة.** النموذج اللغويّ العاديّ له **فم** فقط — يتكلّم في أيّ
> موضوع، ويُخطئ في الحساب. الوكيل (Agent) له **فم ويدان**. اليدان هما
> الأدوات (دوال بايثون حقيقيّة)، و**LangGraph** هو الإطار الذي تستخدمه فرق
> الإنتاج لربط الفم باليدين: رسم بيانيّ صغير من عُقد تُشبكها مرّة، ثمّ تدعوه
> على أيّ سؤال.
>
> اليوم: تبني هذا الرسم بـLangGraph (القسم ٢)، ثمّ ترتقي إلى نظام **متعدّد
> الوكلاء** يُوجِّه فيه **مُشرف** كلّ سؤال إلى **اختصاصيّ** يملك أداة واحدة
> (القسم ٣). هذا هو النمط الذي ينتهي إليه كلّ فريق إنتاجيّ.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this notebook, you can:

- Explain the one line of difference between a chatbot and an agent (hint:
  **who** decides when to call a tool).
- Build a working **ReAct-style agent in LangGraph** — define the state, write
  two nodes, wire a conditional edge, and compile.
- Visualize the compiled graph as a diagram and read its topology.
- Extend the single-agent graph into a **multi-agent system** with a
  *supervisor* that routes each question to the right *specialist*.
- Name the one trade-off a supervisor introduces that a single agent did not have.

<div dir="rtl" align="right">

## أهداف التعلّم

- اشرح السطر الوحيد الذي يفرق بين الروبوت والوكيل (مَن يُقرِّر استدعاء الأداة).
- ابنِ وكيلًا بنمط **ReAct** في **LangGraph**: الحالة، عقدتان، حافّة شرطيّة، ثمّ compile.
- اعرض الرسم البيانيّ المُترجَم كمخطّط واقرأ طوبولوجيّته.
- وسِّع الوكيل الواحد إلى نظام **متعدّد الوكلاء** بـ**مُشرف** يُوجِّه كلّ سؤال إلى **اختصاصيّ**.
- سَمِّ المقايضة الوحيدة التي يُدخلها المُشرف ولم تكن موجودة في الوكيل الواحد.

</div>


## About the model, the tools, and LangGraph

**Model.** `Qwen/Qwen2.5-1.5B-Instruct` — 1.5B parameters, instruction-tuned.
We use the 1.5B here (not 0.5B like D2) because the agent asks the model to
emit a **strict format** (`Thought / Action / Action Input`), and smaller
models drift from that format quickly. Fits in ~3 GB RAM; one step takes
~1 s on Colab T4 or ~3–5 s on Colab CPU.

**Tools.** Two tiny Python functions:
- `tool_calculator(expr)` — safe `eval` of digits and `+ - * / ( ) %`.
- `tool_kb_lookup(key)`   — looks up a key in a 5-entry dict about Olo Academy.

Both are stand-ins. In production, `calculator` becomes a sandboxed math
service, and `kb_lookup` becomes the whole RAG pipeline from D2
(embed → retrieve → return). The agent code does not change.

**LangGraph.** The most-used production framework for agents in late 2026.
You write agents as **graphs**: nodes do work, edges connect them, one
*conditional* edge decides where to go next. The compiled graph is a plain
callable that takes a state dict and returns a state dict — same shape as any
other Python function, except LangGraph gives you tracing, streaming, and
multi-agent orchestration for free.

<div dir="rtl" align="right">

## حول النموذج والأدوات وLangGraph

**النموذج.** `Qwen/Qwen2.5-1.5B-Instruct` — ١٫٥ مليار معامل، مُدرَّب على التعليمات.
نستخدم ١٫٥B لأنّ الوكيل يطلب التزامًا دقيقًا بصيغة ReAct، والنماذج الأصغر تخرج
عن الصيغة سريعًا. يحتاج ~٣ GB من الذاكرة.

**الأدوات.** دالّتان بسيطتان: `tool_calculator` و`tool_kb_lookup`. كلتاهما
بديل عن أدوات إنتاجيّة حقيقيّة (خدمة حساب آمنة، أو مسار RAG كامل كما في D2).

**LangGraph.** أشهر إطار إنتاجيّ للوكلاء في أواخر ٢٠٢٦. الوكيل = **رسم بيانيّ**
من عُقد تربطها حوافّ، إحداها شرطيّة. الرسم المُترجَم مجرّد دالّة تأخذ dict
وتُرجع dict — مع تتبّع وبثّ وتنسيق متعدّد الوكلاء جاهزًا.

</div>


## Setup

<div dir="rtl" align="right">

## الإعداد

</div>


In [1]:
!pip install langchain==0.2

In [1]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import sys, subprocess

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# On Colab, make sure the usual stack is present.
if "google.colab" in sys.modules:
    _pip_install(["transformers>=4.40", "accelerate", "matplotlib", "langgraph>=0.2"])

# Everywhere (including local conda envs): if LangGraph is missing, install it.
# This also catches a stale local env that already has transformers but not LangGraph.
try:
    import langgraph  # noqa: F401
except ModuleNotFoundError:
    print("LangGraph not found in this environment -- installing it now ...")
    _pip_install(["langgraph>=0.2"])

import re, torch
import matplotlib.pyplot as plt
from typing import TypedDict
from transformers import AutoTokenizer, AutoModelForCausalLM
from langgraph.graph import StateGraph, END

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch={torch.__version__}  device={DEVICE}")

torch=2.11.0+cu130  device=cuda


In [2]:
# === Load the chat LLM =============================================================
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm       = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float32).to(DEVICE)
llm.eval()

@torch.no_grad()
def chat(system_message, user_message, max_new_tokens=200, stop=None):
    """One blocking call to the LLM. Returns the generated text (optionally
    truncated at the first `stop` string)."""
    msgs = [{"role":"system","content":system_message},
            {"role":"user",  "content":user_message}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    if stop:
        for s in stop:
            if s in text:
                text = text[:text.index(s)]
    return text.strip()

print(f"Loaded {LLM_NAME}: {sum(p.numel() for p in llm.parameters())/1e6:.0f}M params")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct: 1544M params


## Section 1 — Warm-up: the LLM alone, versus the LLM with a hand

Two tiny demos, no tasks. The whole purpose is to make Section 2's graph
feel inevitable.

1. We ask the LLM a question whose correct answer **requires arithmetic**.
   The LLM has no calculator, so it makes a number up — confidently.
2. We do the arithmetic ourselves in Python, hand the answer to the LLM as a
   fact, and ask again. Now the natural-language answer is correct.

**Section 2 automates step 2** — the LLM, not you, decides when to reach for
Python, and LangGraph orchestrates the loop.

<div dir="rtl" align="right">

## القسم ١ — تسخين: النموذج وحده، ثمّ النموذج بيد

عرضان صغيران بلا مهامّ. نسأل النموذج سؤالًا يتطلّب حسابًا، فيختلق رقمًا بثقة.
ثمّ نُجري الحساب في بايثون ونسلّمه كحقيقة، فيُجيب جوابًا طبيعيًّا صحيحًا.
**القسم ٢ يُؤتمت الخطوة الثانية** — النموذج هو مَن يُقرِّر، وLangGraph يُنسِّق الحلقة.

</div>


In [3]:
# === Demo 1: plain LLM, no tools ===================================================
Q = ("If Olo Academy's AIEP cohort has 40 students and tuition were 2500 SAR each, "
     "what is the total revenue in US dollars at 3.75 SAR per USD?")
print("Q:", Q)
print()
print("LLM alone (no tools):")
print(chat(system_message="You are a helpful assistant.", user_message=Q, max_new_tokens=150))

Q: If Olo Academy's AIEP cohort has 40 students and tuition were 2500 SAR each, what is the total revenue in US dollars at 3.75 SAR per USD?

LLM alone (no tools):
To calculate the total revenue for Olo Academy's AIEP cohort in US dollars, we need to follow these steps:

1. Calculate the total revenue in SAR.
2. Convert the total revenue from SAR to USD using the given exchange rate.

**Step 1: Calculate the total revenue in SAR**

The total number of students is 40, and the tuition fee is 2500 SAR per student. Therefore, the total revenue in SAR is:
\[ \text{Total Revenue (SAR)} = 40 \times 2500 = 100,000 \text{ SAR} \]

**Step 2: Convert the total revenue from SAR to USD**

We know that


In [4]:
# === Demo 2: manual tool call ======================================================
# You do the math in Python, then tell the LLM the result and ask it to answer.
calc_result = (40 * 2500) / 3.75   # SAR -> USD
print(f"Python computed: {calc_result:.2f} USD")
print()
print("LLM with the fact handed to it:")
print(chat(
    system_message="You are a helpful assistant. Use ONLY the facts given to you.",
    user_message=f"Fact: 40 students * 2500 SAR / (3.75 SAR per USD) = {calc_result:.2f} USD.\n\n"
                 f"Question: {Q}",
    max_new_tokens=80,
))

Python computed: 26666.67 USD

LLM with the fact handed to it:
To calculate the total revenue in US dollars for Olo Academy's AIEP cohort based on the given information:

1. The number of students is 40.
2. The cost per student is 2500 SAR.
3. The exchange rate from SAR to USD is 3.75 SAR per USD.

First, we find the total cost in SAR:
\[ \text


> **Think about it.** In Demo 2, **who decided** that a calculation was needed —
> you, or the LLM?
>
> That one word is the whole difference between a chatbot and an agent.
> In Section 2 you move that decision from your Python code into **the LLM
> itself**, orchestrated by a LangGraph graph.

<div dir="rtl" align="right">

> **فكّر.** في العرض ٢: **مَن قرّر** أنّ ثمّة حسبة؟ أنت أم النموذج؟
> هذه الكلمة الواحدة هي الفارق كلّه بين روبوت دردشة ووكيل. القسم ٢ ينقل
> القرار من كودك إلى **النموذج نفسه**، بتنسيق رسم بيانيّ من LangGraph.

</div>


## Section 2 — Build the agent with LangGraph (three tasks)

We hand you four things, pre-written, because they are not the point of the
lesson:

- `tool_calculator(expr)` and `tool_kb_lookup(key)` — the two Python tools.
- `SYSTEM_PROMPT` — the ReAct instructions (strict
  `Thought / Action / Action Input / Final Answer` format).
- `parse_action(text)` — a 5-line regex helper.
- `TOOLS = {"calculator": ..., "kb_lookup": ...}` — the dispatch dict.

**Your job** is the three tasks that turn these pieces into a **compiled
LangGraph agent**:

1. **Task 1** — define `AgentState` (a `TypedDict`) and the two nodes
   `think_node` and `tool_node`.
2. **Task 2** — wire the `StateGraph`: nodes, entry point, a *conditional
   edge* that routes to `END` or back to the tool, and `compile()`.
3. **Task 3** — invoke the compiled agent on a question that needs both
   tools, and on a question it cannot answer.

<div dir="rtl" align="right">

## القسم ٢ — ابنِ الوكيل بـLangGraph (ثلاث مهامّ)

نُسلّمك أربعة أشياء جاهزة: الأداتان، وSYSTEM_PROMPT، وparse_action، وقاموس
TOOLS. **مهمّتك** هي ثلاث خطوات تُحوِّلها إلى وكيل LangGraph مُترجَم:
١) `AgentState` والعُقدتان، ٢) ربط الرسم مع حافّة شرطيّة ثمّ compile،
٣) الاستدعاء على سؤالين.

</div>


> **Think about it (before you read the next cell).** The LLM only ever sees
> the **names** and **one-line descriptions** of the tools — not their source
> code. Why is that enough? What is it reading, and what is it never allowed to see?

<div dir="rtl" align="right">

> **فكّر قبل القراءة.** النموذج لا يرى سوى **أسماء** الأدوات و**وصفها
> المختصر** — لا يرى شيفرتها. لماذا يكفي ذلك؟

</div>


In [5]:
# === Given to you: tools, knowledge base, prompt, parser, dispatch =================

# -- Tool 1: a calculator (safe-ish eval) -------------------------------------------
def tool_calculator(expression: str) -> str:
    expr = expression.strip()
    if not re.fullmatch(r"[0-9+\-*/().%\s]+", expr):
        return "Error: calculator only supports digits and + - * / ( ) % ."
    try:
        return str(eval(expr))
    except Exception as e:
        return f"Error: {e}"

# -- Tool 2: a tiny Olo Academy handbook (stand-in for a RAG pipeline) --------------
KB = {
    "cohort size":        "The AIEP cohort of October 2026 has 40 students.",
    "tuition per student":"Tuition is 2500 SAR per student.",
    "USD exchange rate":  "The current exchange rate is 3.75 SAR per 1 USD.",
    "program length":     "The AIEP bootcamp runs for 8 weeks.",
    "lead instructor":    "The lead instructor is Ghaida Alshehri.",
}
def tool_kb_lookup(key: str) -> str:
    q = key.strip().lower()
    for k, v in KB.items():
        k_lo = k.lower()
        if k_lo in q or q in k_lo:
            return v
    return f"No fact found about {key!r}."

# -- The system prompt the LLM sees on every step -----------------------------------
SYSTEM_PROMPT = """You are a helpful ReAct agent that can use tools to answer questions.

TOOLS AVAILABLE:
- calculator: Evaluates a math expression. Example input: "40 * 2500 / 3.75"
- kb_lookup : Looks up a fact in the Olo Academy handbook. Example input: "cohort size"

FORMAT (strict):
On each step you MUST emit EXACTLY ONE of these two patterns:

Pattern A -- call a tool:
    Thought: <your reasoning in one sentence>
    Action: <calculator OR kb_lookup>
    Action Input: <the string to pass to the tool>

Pattern B -- final answer:
    Thought: <your reasoning in one sentence>
    Final Answer: <the natural-language answer for the user>

After Pattern A, the system fills in an Observation line. Then you continue
from the next Thought.

Never both patterns in the same turn. Never invent tools other than calculator
or kb_lookup. If no tool helps, emit Pattern B with the best answer you can."""

# -- Parser: pulls (action, action_input) out of the model's output -----------------
def parse_action(text):
    m_act = re.search(r"Action\s*:\s*(\S+)", text)
    m_in  = re.search(r"Action Input\s*:\s*(.+)", text)
    if not m_act or not m_in:
        return None, None
    return m_act.group(1).strip(), m_in.group(1).strip().strip('"\'`')

# -- Dispatch table: name -> callable ----------------------------------------------
TOOLS = {"calculator": tool_calculator, "kb_lookup": tool_kb_lookup}

# Quick sanity check
print("calc(2*3)        =", tool_calculator("2*3"))
print("kb('cohort size')=", tool_kb_lookup("cohort size"))

calc(2*3)        = 6
kb('cohort size')= The AIEP cohort of October 2026 has 40 students.


### Task 1 — Define the state and the two nodes

A LangGraph **state** is a `TypedDict` that flows through the graph. Each node
reads it, computes something, and returns a dict of fields to merge back.

Our agent's state has three fields:

- `question`    — the user's original question (never changes).
- `scratchpad`  — the running transcript (grows every step).
- `last_output` — what the LLM said on the most recent think step (so the
  router can look for `"Final Answer:"` in it).

Then two nodes:

- `think_node(state)` — call `chat(SYSTEM_PROMPT, scratchpad, stop=["Observation:"])`
  and return `{"scratchpad": scratchpad + out, "last_output": out}`.
- `tool_node(state)` — parse `last_output`, run `TOOLS[action]`, and return
  `{"scratchpad": scratchpad + "\nObservation: <result>\n"}`.

Return **only the fields you changed** — LangGraph merges them back.

> **Think about it.** A node is just a Python function. What does LangGraph
> add on top of a function that calling it yourself would not give you?

<div dir="rtl" align="right">

### المهمّة ١ — الحالة والعُقدتان

عرّف `AgentState` (TypedDict) بثلاثة حقول نصّيّة: question وscratchpad وlast_output.
ثمّ عقدتين: `think_node` يستدعي النموذج ويُرجع ما تغيّر، و`tool_node` يُحلّل
`last_output` ويُشغّل الأداة ويُلحق Observation.

> **فكّر.** العقدة مجرّد دالّة بايثون. ماذا يُضيف LangGraph فوق مجرّد نداء الدالّة؟

</div>


In [6]:
# ────────────────────────────────────────────────────────────────────
# 1) class AgentState(TypedDict): question: str; scratchpad: str; last_output: str
# 2) think_node: out = chat(SYSTEM_PROMPT, state["scratchpad"], max_new_tokens=200, stop=["Observation:"])
#    return {"scratchpad": state["scratchpad"] + out, "last_output": out}
# 3) tool_node: action, action_input = parse_action(state["last_output"])
# 4) If action in TOOLS: observation = TOOLS[action](action_input), else build an error string.
# 5) Return {"scratchpad": state["scratchpad"] + f"\nObservation: {observation}\n"}.
# 6) Only return the fields you changed; LangGraph merges them into the state.
#
# ١) class AgentState(TypedDict): بثلاثة حقول نصّيّة.
# ٢) think_node: نادِ chat ثمّ أرجع scratchpad الممدَّد وlast_output.
#    أرجع {"scratchpad": state["scratchpad"] + out, "last_output": out}
# ٣) tool_node: نادِ parse_action على last_output.
# ٤) إن كانت الأداة في TOOLS شغّلها، وإلّا ابنِ رسالة خطأ.
# ٥) أرجع {"scratchpad": scratchpad الممدَّد بـObservation}.
# ٦) أرجع الحقول التي غيّرتها فقط — LangGraph يدمجها.
# ────────────────────────────────────────────────────────────────────

# TODO: Define AgentState (TypedDict) and the two node functions.
# مهمة: عرّف AgentState (TypedDict) ودالّتَي العُقدتين.

# === Agent state + the two nodes (think / tool) ====================================

class AgentState(TypedDict):
    question: str
    scratchpad: str
    last_output: str

def think_node(state: AgentState) -> dict:
    out = chat(
        SYSTEM_PROMPT,
        state["scratchpad"],
        max_new_tokens=200,
        stop=["Observation:"],
    )
    return {"scratchpad": state["scratchpad"] + out, "last_output": out}

def tool_node(state: AgentState) -> dict:
    action, action_input = parse_action(state["last_output"])
    if action in TOOLS:
        observation = TOOLS[action](action_input)
    else:
        observation = f"Error: unknown or missing action {action!r}. Use calculator or kb_lookup."
    return {"scratchpad": state["scratchpad"] + f"\nObservation: {observation}\n"}

### Task 2 — Wire the graph and compile

Three edges, one of them conditional. That is the whole topology:

- `think` is the **entry point**.
- From `think`, go to `END` if `last_output` contains `"Final Answer:"`;
  otherwise go to `tool`. (This is the **conditional edge** — the heart of
  the graph.)
- From `tool`, go back to `think` unconditionally.

Compile it: `agent = builder.compile()`. The result is a callable.

> **Think about it.** Why does the "keep going?" decision live on a **conditional
> edge** rather than inside the `tool_node` itself?

<div dir="rtl" align="right">

### المهمّة ٢ — اربط الرسم وترجمه

ثلاث حوافّ، إحداها شرطيّة: `think` نقطة البداية؛ منها إلى `END` إن ظهرت
`"Final Answer:"` في last_output، وإلّا إلى `tool`؛ ومن `tool` دائمًا إلى
`think`. ثمّ `agent = builder.compile()`.

> **فكّر.** لماذا يعيش قرار "هل نُكمل؟" على **حافّة شرطيّة** لا داخل `tool_node`؟

</div>


In [7]:
# ────────────────────────────────────────────────────────────────────
# 1) def should_continue(state): return "end" if "Final Answer:" in state["last_output"] else "tool"
# 2) builder = StateGraph(AgentState)
# 3) builder.add_node("think", think_node); builder.add_node("tool", tool_node)
# 4) builder.set_entry_point("think")
# 5) builder.add_conditional_edges("think", should_continue, {"tool": "tool", "end": END})
# 6) builder.add_edge("tool", "think"); agent = builder.compile()
#
# ١) عرّف should_continue(state) تُرجع "end" أو "tool" حسب last_output.
# ٢) builder = StateGraph(AgentState).
# ٣) أضف العُقدتين think وtool.
# ٤) builder.set_entry_point("think").
# ٥) أضف الحافّة الشرطيّة من think.
# ٦) builder.add_edge("tool", "think") ثمّ builder.compile().
# ────────────────────────────────────────────────────────────────────

# TODO: Build the StateGraph, add the conditional edge from think, and compile into `agent`.
# مهمة: ابنِ StateGraph، أضف الحافّة الشرطيّة من think، وترجم إلى `agent`.
# === Build the ReAct graph: think -> (tool -> think)* -> end =======================

def should_continue(state: AgentState) -> str:
    return "end" if "Final Answer:" in state["last_output"] else "tool"

builder = StateGraph(AgentState)

builder.add_node("think", think_node)
builder.add_node("tool", tool_node)

builder.set_entry_point("think")

builder.add_conditional_edges("think", should_continue, {"tool": "tool", "end": END})

builder.add_edge("tool", "think")

agent = builder.compile()

### Task 3 — Invoke the agent

Call the compiled `agent` on **two questions**:

1. A **compound question** that needs both tools (`kb_lookup` for the facts,
   then `calculator` for the sum). Watch the model pick the tools itself.
2. An **impossible question** (something not in the handbook, e.g. live
   Bitcoin price). Expect either a sensible fallback answer or a
   `GraphRecursionError`.

Pass `config={"recursion_limit": 12}` — the LangGraph equivalent of a
`max_steps` safety cap. Print `final_state["scratchpad"]` so you can see the
whole `Thought / Action / Observation` trace.

<div dir="rtl" align="right">

### المهمّة ٣ — شغّل الوكيل

ناد `agent` على **سؤالين**: سؤال مركّب يحتاج كلتا الأداتين، وسؤال مستحيل
خارج قاعدة المعرفة. استخدم `config={"recursion_limit": 12}` (مكافئ `max_steps`)،
واطبع `final_state["scratchpad"]` لرؤية الأثر الكامل.

</div>


In [8]:
# ────────────────────────────────────────────────────────────────────
# 1) initial = {"question": q, "scratchpad": f"Question: {q}\n", "last_output": ""}
# 2) final_state = agent.invoke(initial, config={"recursion_limit": 12})
# 3) Print final_state["scratchpad"] to see the Thought/Action/Observation trace.
# 4) The final answer is after "Final Answer:" in final_state["last_output"].
# 5) Wrap in try/except — the impossible question may raise GraphRecursionError.
# 6) Loop over QUESTIONS to see both the compound and the impossible one.
#
# ١) initial = {"question": q, "scratchpad": f"Question: {q}\n", "last_output": ""}
# ٢) final_state = agent.invoke(initial, config={"recursion_limit": 12})
# ٣) اطبع final_state["scratchpad"] لرؤية الأثر الكامل.
# ٤) الإجابة بعد "Final Answer:" في last_output.
# ٥) لفّ الاستدعاء في try/except — السؤال المستحيل قد يرفع GraphRecursionError.
# ٦) كرّر على QUESTIONS لرؤية السؤالين.
# ────────────────────────────────────────────────────────────────────


# TODO: Invoke `agent` on both questions; print the final answer and the scratchpad for each.
# مهمة: نادِ `agent` على السؤالين؛ اطبع الإجابة والأثر لكلٍّ منهما.

# === Run the agent on both questions ===============================================
from langgraph.errors import GraphRecursionError

QUESTIONS = [
    "How much total tuition revenue in US dollars does the current AIEP cohort "
    "generate, given the handbook's tuition and current rate?",
    "What is the live price of Bitcoin in Saudi Riyals right now?",
]

for q in QUESTIONS:
    print("=" * 80)
    print("QUESTION:", q)
    print("=" * 80)

    initial = {"question": q, "scratchpad": f"Question: {q}\n", "last_output": ""}

    try:
        final_state = agent.invoke(initial, config={"recursion_limit": 12})

        print("\n--- Trace (scratchpad) ---")
        print(final_state["scratchpad"])

        print("--- Final answer ---")
        last = final_state["last_output"]
        if "Final Answer:" in last:
            print(last.split("Final Answer:", 1)[1].strip())
        else:
            print("(no Final Answer found) ->", last)

    except GraphRecursionError:
        print("\nGraphRecursionError: the agent hit the step limit (12) without reaching a Final Answer.")
    except Exception as e:
        print(f"\nUnexpected error: {type(e).__name__}: {e}")

    print()

QUESTION: How much total tuition revenue in US dollars does the current AIEP cohort generate, given the handbook's tuition and current rate?

--- Trace (scratchpad) ---
Question: How much total tuition revenue in US dollars does the current AIEP cohort generate, given the handbook's tuition and current rate?
Thought: To calculate the total tuition revenue generated by the current AIEP cohort, I need to multiply the number of students by the average tuition per student.
Action: calculator
Action Input: 1680 * 19000
Observation: 31920000
Final Answer: The current AIEP cohort generates a total tuition revenue of $319,200,000 in US dollars.
--- Final answer ---
The current AIEP cohort generates a total tuition revenue of $319,200,000 in US dollars.

QUESTION: What is the live price of Bitcoin in Saudi Riyals right now?

--- Trace (scratchpad) ---
Question: What is the live price of Bitcoin in Saudi Riyals right now?
Thought: To find the current live price of Bitcoin in Saudi Riyals, I woul

### Visualize the compiled graph

LangGraph can print your graph's topology for you. This is the single most
useful debugging habit — before you ship any graph, draw it, and confirm it
matches the picture in your head.

<div dir="rtl" align="right">

### اعرض الرسم المُترجَم

يستطيع LangGraph رسم طوبولوجيّة الرسم البيانيّ. هذه عادة التصحيح الأهمّ:
قبل نشر أيّ رسم، اعرضه وتأكّد أنّه يُطابق الصورة في ذهنك.

</div>


In [9]:
# === Visualize the compiled graph ==================================================
# Mermaid is always available (pure Python). ASCII is the no-dependency fallback.
try:
    print(agent.get_graph().draw_mermaid())
except Exception as e:
    print(f"[mermaid rendering unavailable: {e}]")

# ASCII fallback (always works)
print("\n--- ASCII view ---")
try:
    print(agent.get_graph().draw_ascii())
except Exception:
    # Oldest LangGraph versions name the method differently
    for name, node in agent.get_graph().nodes.items():
        print(f"node: {name}")
    for e in agent.get_graph().edges:
        print(f"edge: {e}")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	think(think)
	tool(tool)
	__end__([<p>__end__</p>]):::last
	__start__ --> think;
	think -. &nbsp;end&nbsp; .-> __end__;
	think -.-> tool;
	tool --> think;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


--- ASCII view ---
node: __start__
node: think
node: tool
node: __end__
edge: Edge(source='__start__', target='think', data=None, conditional=False)
edge: Edge(source='think', target='__end__', data='end', conditional=True)
edge: Edge(source='think', target='tool', data=None, conditional=True)
edge: Edge(source='tool', target='think', data=None, conditional=False)


### Section 2 takeaway

An agent in LangGraph is a tiny loop expressed as a graph. Three parts, and
that is it:

| Part                      | Your code                              |
|---------------------------|----------------------------------------|
| **State**                 | `AgentState(TypedDict)`                |
| **Nodes** (what runs)     | `think_node`, `tool_node`              |
| **The one real decision** | the conditional edge from `think`      |

That's roughly 25 lines of real code. You've just built the production
pattern. Everything added on top of this — streaming, tracing, memory,
multi-agent orchestration — is standard LangGraph machinery you can enable
with one or two more lines.

> **Think about it (read the scratchpad trace above one more time).** At
> **step 3** the LLM "knows" the cohort size and the exchange rate. At
> **step 1** it did not. **Nothing** changed in the model's weights.
> Where did the new knowledge come from?

<div dir="rtl" align="right">

### خلاصة القسم ٢

وكيل LangGraph حلقة صغيرة في صورة رسم بيانيّ، ذات ثلاث قطع فقط: الحالة
والعُقدتين والحافّة الشرطيّة الواحدة. ~٢٥ سطرًا، وقد بنيت النمط الإنتاجيّ
بأكمله. البثّ والتتبّع والذاكرة والتنسيق متعدّد الوكلاء كلّها إضافات تُفعّل
بسطر أو سطرين.

> **فكّر.** في الخطوة ٣ "يعرف" النموذج ما لم يكن يعرفه في الخطوة ١ — ومع
> ذلك **لم يتغيّر** شيء في أوزانه. من أين جاءت المعرفة؟

</div>


## Section 3 — Something extra: multi-agent with a supervisor

The agent in Section 2 has **one brain juggling two tools**. In production,
teams split that up: one *specialist* per domain, plus a *supervisor* that
reads the question and decides who should handle it. It is the same reason
hospitals have triage nurses — the router is cheaper, faster, and more
accurate than making every specialist look at every case.

```
                        Question
                           |
                           v
                  +------------------+
                  |    supervisor    |   (LLM: "math" or "handbook"?)
                  +---+----------+---+
                      |          |
                "math"|          |"handbook"
                      v          v
            +-----------+   +-----------+
            | math      |   | handbook  |
            | specialist|   | specialist|
            |(calc only)|   |(KB only)  |
            +-----+-----+   +-----+-----+
                  |               |
                  v               v
                END             END
```

Each specialist is a **node** in a graph, not a free function. The
`StateGraph` orchestrates **which** node runs for **which** question. That is
LangGraph's actual superpower — try wiring the picture above with a plain
Python `for` loop and you immediately feel the pain.

<div dir="rtl" align="right">

## القسم ٣ — إضافة: نظام متعدّد الوكلاء بمُشرف

في القسم ٢ وكيل واحد يُدير أداتين. في الإنتاج تُقسَّم المسؤوليّة: **اختصاصيّ**
لكلّ مجال، **ومشرف** يقرأ السؤال ويُقرِّر مَن يتولّاه. تمامًا كممرّضة الفرز في
المستشفيات: المُوجِّه أسرع وأرخص وأدقّ من إشراك كلّ اختصاصيّ في كلّ حالة.

كلّ اختصاصيّ **عقدة** في رسم بيانيّ، لا دالّة مستقلّة. وLangGraph يُنسِّق
أيّ عقدة تعمل لأيّ سؤال. هذه قوّته الحقيقيّة — جرّب نفس الصورة بـ`for`
بسيطة وستشعر بالفوضى.

</div>


### Task 1 — Build the two specialist nodes

Each specialist is a plain Python function that takes the shared state and
returns `{"final_answer": "..."}`. **No ReAct loop** this time — a specialist
owns exactly one tool, so the LLM only has to pick the tool's input and
compose the final sentence. Two LLM calls per specialist, no graph needed
inside.

- **`math_specialist(state)`** — ask the LLM for ONE calculator expression,
  run `tool_calculator`, then ask the LLM to turn the result into a
  one-sentence answer.
- **`handbook_specialist(state)`** — ask the LLM for ONE handbook key,
  run `tool_kb_lookup`, then ask the LLM to turn the result into a
  one-sentence answer.

> **Think about it.** Why no `max_steps` here? What property of a specialist
> makes the recursion-limit safety cap unnecessary?

<div dir="rtl" align="right">

### المهمّة ١ — ابنِ عُقدتَي الاختصاصيّين

كلّ اختصاصيّ دالّة بسيطة تأخذ الحالة وتُرجع `{"final_answer": "..."}`. **لا
حلقة ReAct** هنا — الاختصاصيّ يملك أداة واحدة، فعلى النموذج فقط اختيار
مدخلها وصياغة الجواب. ندائان للنموذج فقط لكلّ اختصاصيّ.

- **`math_specialist(state)`** — اطلب من النموذج تعبيرًا حسابيًّا واحدًا، شغّل الحاسبة، ثمّ اطلب جوابًا بجملة واحدة.
- **`handbook_specialist(state)`** — اطلب من النموذج مفتاح بحث واحد، شغّل kb_lookup، ثمّ اطلب جوابًا بجملة واحدة.

> **فكّر.** لماذا لا نحتاج هنا إلى `max_steps`؟ ما خاصّيّة الاختصاصيّ التي تجعل سقف الأمان غير ضروريّ؟

</div>


In [10]:
# ────────────────────────────────────────────────────────────────────
# 1) Define SupervisorState(TypedDict) with: question (str), route (str), final_answer (str).
# 2) math_specialist: call chat(sys1, user1) asking for ONE calculator expression; grab the first line.
# 3)                  Run tool_calculator on it. Then call chat(sys2, user2) to compose a one-sentence answer.
# 4)                  Return {"final_answer": that sentence}.
# 5) handbook_specialist: same shape, but ask for ONE handbook key, then run tool_kb_lookup, then compose.
# 6) Both specialists only read state["question"] and return {"final_answer": ...}.
#
# ١) عرّف SupervisorState(TypedDict) بثلاثة حقول: question وroute وfinal_answer.
# ٢) math_specialist: نادِ chat ليُعطي تعبيرًا حسابيًّا واحدًا؛ خذ أوّل سطر.
# ٣)                  شغّل tool_calculator، ثمّ نادِ chat ليُعطي جملة الجواب.
# ٤)                  أرجع {"final_answer": تلك الجملة}.
# ٥) handbook_specialist: نفس الشكل لكن المفتاح بدل التعبير، وtool_kb_lookup بدل الحاسبة.
# ٦) كلا الاختصاصيّين يقرآن state["question"] فقط ويُرجعان {"final_answer": ...}.
# ────────────────────────────────────────────────────────────────────

# TODO: Define SupervisorState and the two specialist node functions.
# مهمة: عرّف SupervisorState ودالّتَي الاختصاصيّين.
# === Supervisor state + the two specialist nodes ===================================

class SupervisorState(TypedDict):
    question: str
    route: str
    final_answer: str

def math_specialist(state: SupervisorState) -> dict:
    q = state["question"]

    # Step 1: ask the LLM for ONE calculator expression
    sys1 = ("You are a math assistant. Convert the user's question into ONE arithmetic "
            "expression using only digits and + - * / ( ) . "
            "Output ONLY the expression on a single line, nothing else.")
    expr_raw = chat(sys1, q, max_new_tokens=40).strip()
    expr = expr_raw.splitlines()[0].strip().strip("`").rstrip("=").strip() if expr_raw else ""

    # Step 2: run the calculator tool
    result = tool_calculator(expr)

    # Step 3: ask the LLM to compose a one-sentence answer from the facts
    sys2 = "You are a helpful assistant. Use ONLY the facts given to you. Answer in ONE sentence."
    user2 = (f"Question: {q}\n"
             f"Calculation: {expr} = {result}\n"
             f"Write a one-sentence answer.")
    answer = chat(sys2, user2, max_new_tokens=80)

    return {"final_answer": answer}

def handbook_specialist(state: SupervisorState) -> dict:
    q = state["question"]

    # Step 1: ask the LLM for ONE handbook key
    keys = ", ".join(KB.keys())
    sys1 = (f"You look up facts in the Olo Academy handbook. Available keys: {keys}. "
            "Pick the ONE key that best answers the question. "
            "Output ONLY the key on a single line, nothing else.")
    key_raw = chat(sys1, q, max_new_tokens=20).strip()
    key = key_raw.splitlines()[0].strip().strip("`\"'") if key_raw else ""

    # Step 2: run the handbook lookup tool
    fact = tool_kb_lookup(key)

    # Step 3: ask the LLM to compose a one-sentence answer from the fact
    sys2 = "You are a helpful assistant. Use ONLY the facts given to you. Answer in ONE sentence."
    user2 = (f"Fact: {fact}\n"
             f"Question: {q}\n"
             f"Write a one-sentence answer using only the fact.")
    answer = chat(sys2, user2, max_new_tokens=80)

    return {"final_answer": answer}

### Task 2 — Build the supervisor graph

Three nodes (`supervisor`, `math`, `handbook`), one conditional edge, two
terminal edges to `END`.

- `supervisor_node(state)` — ask the LLM one yes/no-ish question:
  "`math`" or "`handbook`"? Save the answer into `state["route"]`.
- `router(state)` returns `state["route"]` so the conditional edge knows
  where to go.
- Wire: `supervisor → (math | handbook) → END`.

> **Think about it.** The supervisor asks the LLM **once**, then hands off.
> What did we gain compared to the Section 2 agent? What did we lose? (Try
> the compound revenue question through this graph and watch what happens.)

<div dir="rtl" align="right">

### المهمّة ٢ — ابنِ رسم المُشرف

ثلاث عُقد (`supervisor` و`math` و`handbook`)، حافّة شرطيّة واحدة، وحافّتان
إلى `END`.

- `supervisor_node(state)` — اسأل النموذج سؤالًا واحدًا تقريبًا نعم/لا: "math" أم "handbook"؟
- `router(state)` يُرجع `state["route"]` للحافّة الشرطيّة.
- الربط: `supervisor → (math | handbook) → END`.

> **فكّر.** المُشرف يسأل النموذج **مرّة** ثمّ يُسلّم. ما الذي كسبته مقارنة
> بوكيل القسم ٢؟ وما الذي خسرته؟ (جرّب سؤال الإيرادات المركّب عبر هذا الرسم.)

</div>


In [11]:
# ────────────────────────────────────────────────────────────────────
# 1) supervisor_node: ask chat() for ONE word -- "math" or "handbook".
# 2) Normalize: route = "math" if "math" in reply.lower() else "handbook".
# 3) Return {"route": route}.
# 4) router(state): return state["route"]  # drives the conditional edge
# 5) builder = StateGraph(SupervisorState); add_node("supervisor", ...); add_node("math", math_specialist); add_node("handbook", handbook_specialist)
# 6) set_entry_point("supervisor"); add_conditional_edges("supervisor", router, {"math":"math","handbook":"handbook"})
# 7) add_edge("math", END); add_edge("handbook", END); multi_agent = builder.compile()
#
# ١) supervisor_node: اطلب من chat() كلمة واحدة -- "math" أو "handbook".
# ٢) اضبط: route = "math" إذا احتوت الإجابة على "math" وإلّا "handbook".
# ٣) أرجع {"route": route}.
# ٤) router(state): يُرجع state["route"] للحافّة الشرطيّة.
# ٥) builder = StateGraph(SupervisorState) ثمّ أضف العُقد الثلاث.
# ٦) set_entry_point("supervisor") ثمّ add_conditional_edges من المُشرف إلى math/handbook.
# ٧) أضف الحافّتين إلى END ثمّ multi_agent = builder.compile().
# ────────────────────────────────────────────────────────────────────

# TODO: Define supervisor_node + router, build the graph with the conditional edge, and compile `multi_agent`.
# مهمة: عرّف supervisor_node وrouter، ابنِ الرسم بالحافّة الشرطيّة، وترجم إلى `multi_agent`.
# === Supervisor + router + multi-agent graph =======================================

def supervisor_node(state: SupervisorState) -> dict:
    sys_msg = ("You are a supervisor that routes a question to ONE specialist. "
               "Reply with exactly ONE word: 'math' if the question needs a calculation, "
               "or 'handbook' if it only needs a fact from the Olo Academy handbook. "
               "Output only that one word.")
    reply = chat(sys_msg, state["question"], max_new_tokens=5)
    route = "math" if "math" in reply.lower() else "handbook"
    return {"route": route}

def router(state: SupervisorState) -> str:
    return state["route"]

builder = StateGraph(SupervisorState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("math", math_specialist)
builder.add_node("handbook", handbook_specialist)

builder.set_entry_point("supervisor")

builder.add_conditional_edges("supervisor", router, {"math": "math", "handbook": "handbook"})

builder.add_edge("math", END)
builder.add_edge("handbook", END)

multi_agent = builder.compile()

### Run the multi-agent system on three questions

Three questions of increasing awkwardness. Watch the `route` the supervisor
picks on each — and the one where routing alone is not enough.

<div dir="rtl" align="right">

### شغّل النظام متعدّد الوكلاء على ثلاثة أسئلة

ثلاثة أسئلة بتعقيد متدرّج. راقب `route` الذي يختاره المُشرف — والسؤال الذي
لا يكفيه التوجيه وحده.

</div>


In [12]:
# === Run the multi-agent system ====================================================
MA_QUESTIONS = [
    "How many students are in the current AIEP cohort?",                 # should route -> handbook
    "What is forty times two thousand five hundred divided by three point seven five?",  # -> math
    "How much total tuition revenue in US dollars does the current AIEP cohort "         # compound; routing is imperfect
    "generate, given the handbook's tuition and current rate?",
]

for q in MA_QUESTIONS:
    print("=" * 70)
    print(f"Q: {q}")
    print("=" * 70)
    initial = {"question": q, "route": "", "final_answer": ""}
    final_state = multi_agent.invoke(initial)
    print(f"Route chosen : {final_state['route']}")
    print(f"Final answer : {final_state['final_answer']}")
    print()

# Also draw the supervisor graph so you can see it next to Section 2's.
print("--- Supervisor graph ---")
try:
    print(multi_agent.get_graph().draw_mermaid())
except Exception as e:
    print(f"[mermaid unavailable: {e}]")

Q: How many students are in the current AIEP cohort?
Route chosen : handbook
Final answer : There are 40 students in the current AIEP cohort for October 2026.

Q: What is forty times two thousand five hundred divided by three point seven five?
Route chosen : math
Final answer : The result of dividing forty times two thousand five hundred by three point seven five, calculated as 100,000 / 3.75, equals approximately 26,666.67.

Q: How much total tuition revenue in US dollars does the current AIEP cohort generate, given the handbook's tuition and current rate?
Route chosen : math
Final answer : The calculation results in zero because there is no information provided about the number of students enrolled or their tuition fees for the AIEP cohort.

--- Supervisor graph ---
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	math(math)
	handbook(handbook)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	supervi

> **Think about it (the final reflection).** The compound revenue question
> cannot really be answered by **one** specialist — it needs both. The
> supervisor had to pick one anyway, and the answer suffered.
>
> **What did we gain** with the supervisor?
> → Clear separation of concerns, each specialist can be tested in isolation,
>   no `max_steps` safety cap needed, cheaper (one LLM step for routing + one
>   or two for the specialist, vs. up to twelve for the full agent).
>
> **What did we lose?**
> → Compound tasks that need **both** specialists. For those, you either go
>   back to the single-agent ReAct of Section 2 **or** build a richer graph
>   where specialists can hand off to each other (an "agentic team" pattern).

<div dir="rtl" align="right">

> **فكّر (التأمّل الأخير).** سؤال الإيرادات المركّب لا يحلّه اختصاصيّ **واحد**
> بالكامل. اضطُرّ المُشرف إلى اختيار واحد، فتأذّى الجواب.
>
> **ما كسبته بالمُشرف؟** فصلٌ واضح للمسؤوليّات، اختبار الاختصاصيّين منفصلين،
> لا سقف `max_steps`، وتكلفة أقلّ.
>
> **ما خسرته؟** المهامّ التي تحتاج **كلا** الاختصاصيّين. لتلك إمّا تعود إلى
> وكيل القسم ٢، أو تبني رسمًا أغنى يسمح للاختصاصيّين بتسليم المهمّة لبعضهم.

</div>


## Wrap-up

| What you built                 | Section 2 (one agent)                       | Section 3 (supervisor + 2 specialists)       |
|--------------------------------|---------------------------------------------|----------------------------------------------|
| Framework                      | LangGraph `StateGraph`                      | LangGraph `StateGraph`                       |
| State                          | `AgentState` (question/scratchpad/last)     | `SupervisorState` (question/route/answer)    |
| Nodes                          | `think_node`, `tool_node`                   | `supervisor_node`, `math`, `handbook`        |
| Decision point                 | conditional edge `think → (tool | END)`     | conditional edge `supervisor → (math | hb)`  |
| Shines on                      | compound questions needing many tools       | clear single-domain questions                |
| LLM calls per question         | up to recursion_limit (~5–12)               | exactly 3 (route + input + answer)           |

### Where you can take this next

- **Add a memory checkpointer** so the agent remembers prior turns of the
  same conversation (e.g. "and per week?" after the revenue question).
  LangGraph has `MemorySaver`: initialize it, pass it to `compile(checkpointer=...)`,
  then invoke with `config={"configurable": {"thread_id": "chat-1"}, "recursion_limit": 12}`.
  Second invocation on the same `thread_id` carries the state forward.
- **Let specialists hand off to each other** (the "agentic team" pattern) —
  swap the two terminal `END` edges for conditional edges back into the
  supervisor, so a specialist can say "I need help" and be rerouted.
- **Stream tokens** with `agent.stream(initial)` instead of `.invoke()` to
  see each node fire in real time.
- **Trace runs** by setting `LANGSMITH_API_KEY` — every graph step shows up
  on the LangSmith dashboard with inputs, outputs, and timings.

**Next (D4):** same Qwen base — but you fine-tune it with **LoRA** on 10 Olo
Academy FAQs and quantize it to **4-bit** so it fits in a tiny GPU. Then you
compare: did the fine-tune change what the model knows, or just how it talks?

<div dir="rtl" align="right">

## الخلاصة

| ما بنيته                | القسم ٢ (وكيل واحد)                | القسم ٣ (مُشرف + اختصاصيّان)        |
|--------------------------|--------------------------------------|---------------------------------------|
| الإطار                   | LangGraph                            | LangGraph                             |
| الحالة                   | AgentState (ثلاثة حقول)             | SupervisorState (ثلاثة حقول)         |
| العُقد                   | think + tool                         | supervisor + math + handbook          |
| نقطة القرار              | حافّة شرطيّة من think               | حافّة شرطيّة من supervisor            |
| يتفوّق على               | الأسئلة المركّبة متعدّدة الأدوات     | الأسئلة الواضحة ذات المجال الواحد      |

### الخطوة التالية

- **أضف ذاكرة (`MemorySaver`)** حتّى يتذكّر الوكيل الدور السابق في نفس المحادثة.
- **اسمح للاختصاصيّين بالتسليم فيما بينهم** (نمط "الفريق العاملي").
- **استخدم `agent.stream()`** لرؤية كلّ عقدة تعمل لحظة بلحظة.
- **تتبّع التشغيل** بـLangSmith.

**القادم (D4):** نفس Qwen — لكن تُدرّبه بـLoRA على ١٠ أسئلة ثمّ تُكمّمه إلى ٤ بتّات.

</div>
